# Auditing one failure: the table-retrieval problem

We're going to take **one** question our agent gets wrong and follow it all the
way down — the question, the "golden" answer, the real SEC filing, our agent's
live trace, and exactly what it pulls back — so *you* can see where it breaks
instead of taking my word for it.

The example: **"What geographies does American Express operate in, as of 2022?"**
The answer is right there in the filing (United States, EMEA, APAC, LACC) — but
it lives in a **table**, and our chunker only knows how to read prose. Watch what
happens.

> Run the cells top to bottom. The agent and retrieval cells make real API calls,
> so each takes a few seconds. Want to audit a *different* miss instead? Change
> `QID` in the next cell (e.g. `"00585"`, `"00222"`).

In [ ]:
import textwrap

from sec_filings.inspection import (
    eval_questions,
    retrieve_ranked,
    best_gold_overlap,
    filing_chunks,
)
from sec_filings.agent.loop import run_agent

QID = "financebench_id_01028"   # <-- the failure we're auditing (AXP geographies)


def find_q(qid: str):
    for q in eval_questions():
        if q.question_id == qid or q.question_id[-5:] == qid[-5:]:
            return q
    raise SystemExit(f"{qid} not found in the eval set")


q = find_q(QID)
print("QUESTION :", q.question)
print("COMPANY  :", q.ticker, "FY", q.fiscal_year, "|", q.doc_name)
print("TYPE     :", q.question_type)

## Step 1 — the question and the "golden" answer

Every FinanceBench question ships with two things we grade against:

- a **gold answer** — the short, correct response, and
- **gold evidence** — the exact passage from the filing the answer comes from.

Look closely at the gold evidence below. It's a **table** — geographic regions
down the side, years across the top, revenue in the cells. But by the time it's
been pulled out of the filing's HTML, it's been *flattened into a stream of text*:
the region names ("United States", "EMEA", "APAC", "LACC") end up on their own
lines, and the dollar figures end up in a separate run far below them.

That flattening is the whole bug in miniature. Keep it in mind.

In [ ]:
print("GOLD ANSWER:")
print("  ", q.gold_answer)
print()
print("GOLD EVIDENCE  (the benchmark's cited proof — note it's a flattened table):")
print("-" * 70)
print(q.gold_evidence)

## Step 2 — go look at the real thing on SEC.gov

Before we touch the agent, see the source for yourself. The cell below ingests
the actual American Express 2022 10-K from SEC EDGAR (the same fetch our indexer
uses) and prints a **direct link** to the filing.

Open it, and find the geographic-region table — it's in **Item 8 (Financial
Statements)**, in the segment/geography note. On SEC.gov it renders as a clean
grid: regions in the left column, a *Total revenues net of interest expense* row
with `$41,396 / $4,871 / $3,835 / $2,917` across the regions. Perfectly readable
to a human. Hold that mental picture — our pipeline never gets to see it that way.

In [ ]:
# Ingest + chunk the filing once (deterministic — these are the SAME chunks that
# went into the search index). We reuse `chunks` again in Step 5.
filing, chunks = filing_chunks(q.ticker, q.fiscal_year)

acc = filing.accession_number
edgar = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{filing.cik}/{acc.replace('-', '')}/{acc}-index.htm"
)
print("Company   :", filing.company_name)
print("Accession :", acc)
print("Chunks in this filing:", len(chunks))
print()
print("Open the real 10-K (look in Item 8 for the geographic-region table):")
print(" ", edgar)

## Step 3 — watch our agent try to answer it

Now the live agent. `run_agent` runs the real hand-written tool-use loop at
temperature 0 (deterministic — so this trace matches what the eval scored). It
prints every move: what it's **thinking**, which **tool** it calls and with what
query, which **passages** come back, and its **final answer**.

The thing to watch: in the list of retrieved passages, does *any* of them mention
**EMEA** or **LACC**? That's the tell for whether the agent ever even saw the
table it needed.

In [ ]:
trace = run_agent(q.question)   # live API calls — gives it a few seconds


def show_trace(tr) -> None:
    for s in tr.steps:
        if s.type == "reasoning":
            print("THINKING:", textwrap.shorten(s.content, 320))
            print()
        elif s.type == "tool_call":
            args = {k: v for k, v in (s.tool_input or {}).items()}
            print(f"CALLS  {s.tool_name}({args})")
        elif s.type == "tool_result":
            if s.tool_name == "retrieve" and isinstance(s.tool_output, dict):
                passages = s.tool_output.get("passages", [])
                print(f"  -> got {len(passages)} passages back:")
                for i, p in enumerate(passages, 1):
                    txt = p.get("text", "")
                    flag = "   <== mentions EMEA/LACC!" if ("EMEA" in txt or "LACC" in txt) else ""
                    print(f"       {i:>2}. [{p.get('item')}]  {p.get('chunk_id')}{flag}")
            else:
                print("  ->", textwrap.shorten(str(s.tool_output), 220))
            print()
        elif s.type == "answer":
            print("=" * 70)
            print("AGENT'S FINAL ANSWER:")
            print(textwrap.fill(s.content, 88))


show_trace(trace)
print()
print("-" * 70)
print("GOLD ANSWER:", q.gold_answer)

## Step 4 — what the retriever actually hands over

The agent is only as good as what it can find. This cell runs the retriever
directly on the question and shows the **top 8 chunks** it returns — the exact
shortlist the agent reads from.

Two columns to read:

- **overlap** — what fraction of the gold table's *numbers* show up in that chunk.
  High overlap = "this chunk contains the figures the answer needs."
- the **EMEA/LACC flag** — does the chunk even contain the region names?

If the top 8 are all low-overlap prose with no region names, the table simply
isn't in the agent's field of view. It's answering blind.

In [ ]:
hits = retrieve_ranked(
    q.question,
    ticker=q.ticker,
    fiscal_year=q.fiscal_year,
    top_k=8,
    gold_evidence=q.gold_evidence,
)

print(f"Top {len(hits)} chunks the retriever returns for the raw question:\n")
for h in hits:
    has_region = ("EMEA" in h.text or "LACC" in h.text)
    region = "  <== has EMEA/LACC" if has_region else ""
    print(
        f"  rank {h.rank:>2}  [{h.item:<8}] "
        f"overlap={h.gold_overlap:>4.0%}  gold#s_found={h.gold_numbers_hit}{region}"
    )
print("\n(overlap = fraction of the gold table's numbers present in that chunk)")

## Step 5 — the smoking gun: where did the table go?

So the retriever didn't surface it. Two possible reasons, and they call for
*different* fixes:

1. **The table is chunked fine, but the search query just doesn't match it** —
   then it's sitting at some deep rank we never look at. Fixable by retrieving
   deeper or with a better query.
2. **The table got shredded during chunking** — the region names and their
   numbers landed in different chunks, or got buried inside a giant wall of other
   numbers — so no single chunk is a clean "geographic regions" hit. This is the
   real table-aware-chunking problem.

This cell tells us which. First it finds **every** chunk in the whole filing that
even mentions EMEA/LACC and prints them, so you can see with your own eyes how the
region names sit relative to their numbers. Then `best_gold_overlap` searches 40
deep and reports the *best-covering* chunk's rank — if that's far past 8, or the
overlap is low, the evidence is effectively unreachable.

In [ ]:
region_chunks = [c for c in chunks if ("EMEA" in c.text or "LACC" in c.text)]
print(f"{len(region_chunks)} of {len(chunks)} chunks in the ENTIRE filing even mention EMEA/LACC.\n")

for c in region_chunks:
    print(f"--- {c.chunk_id}   [{c.item}]   ({c.token_count} tokens) ---")
    print(textwrap.shorten(c.text.replace("\n", " "), 700))
    print()

print("=" * 70)
rank, overlap = best_gold_overlap(
    q.question,
    ticker=q.ticker,
    fiscal_year=q.fiscal_year,
    gold_evidence=q.gold_evidence,
    search_depth=40,
)
print(f"Searching 40 deep, the BEST chunk for the gold numbers is at rank {rank}, "
      f"covering {overlap:.0%} of them.")
print("The agent only reads the top 8 — so anything deeper than that, it never sees.")

## Step 6 — your turn: try to coax the table out

Now play with it. Edit `MY_QUERY` below and re-run — try to write a query that
pulls the geographic table into the top 8. Throw the region names at it, the
exact row label ("Total revenues net of interest expense"), the dollar figures,
whatever. See how hard (or impossible) it is to get a clean hit.

This is the part that makes the problem *felt*: even when you, a human who knows
the answer, hand-craft the perfect query, the shredded table is hard to retrieve
as one coherent piece. That's what "table-aware chunking" would fix — by keeping
the region names glued to their numbers and labeling the chunk as a geographic
table in the first place.

In [ ]:
MY_QUERY = "geographic regions United States EMEA APAC LACC total revenues net of interest expense"

mine = retrieve_ranked(
    MY_QUERY,
    ticker=q.ticker,
    fiscal_year=q.fiscal_year,
    top_k=8,
    gold_evidence=q.gold_evidence,
)
print(f"Query: {MY_QUERY!r}\n")
for h in mine:
    has_region = ("EMEA" in h.text or "LACC" in h.text)
    region = "  <== has EMEA/LACC" if has_region else ""
    print(f"  rank {h.rank:>2}  [{h.item:<8}] overlap={h.gold_overlap:>4.0%}{region}")

## What you just saw

- The answer **is in the filing** — but as a **table**, and our chunker reads
  text sentence-by-sentence, so the table got flattened and scattered.
- Because of that, the chunk that *should* be a clean "geographic regions" hit
  either doesn't exist or is buried past rank 8, so the agent never reads it.
- No prompt tweak fixes this — the agent can't reason over evidence it never
  receives. The fix has to happen **upstream, at chunking time**: pull the
  structured tables out, keep each row's label attached to its numbers, and index
  each table as its own labeled chunk.

That's the "table-aware retrieval" work — and now you've seen, end to end,
exactly why it's the highest-leverage thing on the list.